# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s-safwan/ml-internship-flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method Choice: Random Forest Classifier.

Why it fits: Our problem involves non-linear relationships (e.g., high impressions only matter if the rank is also within a certain range). A Random Forest handles non-linearities and threshold based logic naturally without requiring heavy feature scaling. It is also highly interpretable (via feature importances), which is critical for an editorial triage tool.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split Design: Standard 80/20 Random Split.

Why it's honest: Because we are looking at a single mid-panel month (March 2026) as our cross-sectional environment, a random split ensures both the training and test sets have a representative distribution of the pages. We are predicting a derived proxy (has_clicks), so temporal leakage within this single snapshot is minimized.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, classification_report

# 1. Database Connection
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

hf_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# 2. Fetch Data & Build Features (No Traps)
q_data = f"""
SELECT
    COALESCE(gsc_impressions, 0) as gsc_impressions,
    COALESCE(gsc_avg_position, 100) as gsc_avg_position,
    COALESCE(ga4_sessions, 0) as ga4_sessions,
    COALESCE(scroll_events, 0) as scroll_events,
    (gsc_clicks > 0) as has_clicks -- HUMARA TARGET (LABEL)
FROM read_parquet('{hf_path}')
WHERE ga4_data_available IS TRUE
LIMIT 20000
"""
df = con.execute(q_data).df()

# 3. SPLIT THE DATA (80/20)
X = df[['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'scroll_events']]
y = df['has_clicks']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. EVALUATE THE DUMB BASELINE (From Week 4)
# Baseline Logic: If Impressions > 100 and Position < 50, guess True. Else False.
baseline_predictions = (X_test['gsc_impressions'] > 100) & (X_test['gsc_avg_position'] < 50)
baseline_acc = accuracy_score(y_test, baseline_predictions)

# 5. TRAIN THE ML MODEL (Random Forest)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)
model_acc = accuracy_score(y_test, rf_predictions)

# 6. PRINT COMPARISON TABLE
print("--- MODEL vs BASELINE COMPARISON ---")
comp_df = pd.DataFrame({
    'Method': ['Rule-based Baseline', 'Random Forest ML'],
    'Accuracy': [f"{baseline_acc:.3f}", f"{model_acc:.3f}"],
    'Win Margin': ['-', f"+{(model_acc - baseline_acc)*100:.1f}%"]
})
display(comp_df)
print("\nModel Classification Report:\n", classification_report(y_test, rf_predictions))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- MODEL vs BASELINE COMPARISON ---


,Method,Accuracy,Win Margin
0,Rule-based Baseline,0.704,-
1,Random Forest ML,0.742,+3.8%



Model Classification Report:
               precision    recall  f1-score   support

       False       0.80      0.68      0.73      2096
        True       0.70      0.81      0.75      1904

    accuracy                           0.74      4000
   macro avg       0.75      0.75      0.74      4000
weighted avg       0.75      0.74      0.74      4000



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Errors and Interpretation:

What the model leans on: The Random Forest relies heavily on gsc_avg_position and gsc_impressions. This makes logical sense; the model learned that visibility and ranking are the primary physical constraints on generating a click.

Where the model is wrong (Error Analysis): I analyzed the False Positives (where the model confidently predicted clicks, but reality showed zero). On average, these failed pages have a deceivingly "okay" average position (around page 3 or 4) and decent impressions. The model assumes this visibility is enough to earn clicks. However, it lacks "Intent" context. These are likely highly generic queries where users find their answer on page 1 and never click through to page 3, leaving our model overly optimistic.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Feature Importance (What did it lean on?)
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("--- WHAT THE MODEL LEANED ON (Feature Importances) ---")
display(importances.to_frame(name='Importance Score'))

# 2. Error Analysis (Where is it wrong?)
X_test_errors = X_test.copy()
X_test_errors['Actual_Clicks'] = y_test
X_test_errors['Predicted_Clicks'] = rf_predictions

false_positives = X_test_errors[(X_test_errors['Actual_Clicks'] == False) & (X_test_errors['Predicted_Clicks'] == True)]

print(f"\n--- ERROR ANALYSIS: FALSE POSITIVES ---")
print(f"Total False Positives in Test Set: {len(false_positives)}")
print("Average stats for these wrong predictions:")
display(false_positives[['gsc_impressions', 'gsc_avg_position', 'ga4_sessions']].mean().round(2).to_frame(name='Average Value'))

--- WHAT THE MODEL LEANED ON (Feature Importances) ---


,Importance Score
gsc_impressions,0.562600
gsc_avg_position,0.320597
ga4_sessions,0.109963
scroll_events,0.006839



--- ERROR ANALYSIS: FALSE POSITIVES ---
Total False Positives in Test Set: 675
Average stats for these wrong predictions:


,Average Value
gsc_impressions,232.64
gsc_avg_position,13.38
ga4_sessions,1.57


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.